In [2]:
# =====================================================
# 0. KAGGLE-SAFE ENVIRONMENT – ONE COMMAND FIX
# =====================================================
import os, warnings, torch, numpy as np
os.environ['NUMBA_DISABLE_JIT'] = '1'
warnings.filterwarnings("ignore", category=UserWarning)

# 1. Install **compatible** versions in ONE go (critical!)
!pip install --no-cache-dir -q \
    numpy==1.26.4 \
    scipy==1.13.1 \
    scikit-learn==1.5.1 \
    albumentations==1.4.8 \
    opencv-python-headless==4.10.0.84 \
    numba==0.60.0 \
    llvmlite==0.43.0 \
    torch==2.3.1+cu121 torchvision==0.18.1+cu121 torchaudio==2.3.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

# 2. Verify the ABI is consistent
print("NumPy version :", np.__version__)
print("SciPy version :", __import__('scipy').__version__)
print("sklearn version:", __import__('sklearn').__version__)
print("Device        :", "cuda" if torch.cuda.is_available() else "cpu")

ModuleNotFoundError: No module named 'torch'

In [3]:
# Gesture Recognition: Focus on Classes with Any Positive F1
# Install required libs
import os
import scipy
import numpy as np
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SubsetRandomSampler
import torchvision
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.amp import GradScaler, autocast
from tqdm import tqdm
from IPython.display import FileLink, display

torch.backends.cudnn.benchmark = True

# ----------------------
# Settings
DATA_DIR    = '/kaggle/input/sign-language-dataset-wlasl-videos/dataset/SL'
MAX_FRAMES  = 8
BATCH_SIZE  = 32
LR_HEAD     = 1e-4
LR_FINE     = 1e-5
EPOCH_HEAD1 = 10    # quick head-only to gauge per-class F1
EPOCH_HEAD2 = 60    # head-only for final training
EPOCH_FINE  = 40    # fine-tune epochs
VAL_SIZE    = 0.2
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# ----------------------
# Video Transform: robust to bad frames
class VideoTransform:
    def __init__(self, max_frames=MAX_FRAMES):
        self.max_frames = max_frames
        self.aug = A.Compose([
            A.RandomResizedCrop( size = (112, 112), scale=(0.8, 1.0), p=1.0),  # NEW - works
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.5),
            A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
            A.ColorJitter(p=0.5),
            A.Normalize(mean=(0.45,0.45,0.45), std=(0.225,0.225,0.225)),
            ToTensorV2()
        ])

    def __call__(self, clip):
        frames = []
        for i in range(min(self.max_frames, clip.shape[0])):
            frm = clip[i]
            try:
                rgb = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
                t = self.aug(image=rgb)['image']  # (C,H,W)
                if t.ndim == 3:
                    frames.append(t)
            except:
                continue
        if not frames:
            return torch.zeros((3, self.max_frames, 112, 112), dtype=torch.float)
        if len(frames) < self.max_frames:
            frames = (frames * ((self.max_frames // len(frames)) + 1))[:self.max_frames]
        else:
            frames = frames[:self.max_frames]
        stacked = torch.stack(frames)  # (T,C,H,W)
        return stacked.permute(1,0,2,3)  # (C,T,H,W)

# ----------------------
# Dataset wrapper
class GestureDataset(Dataset):
    def __init__(self, classes, transform=None):
        self.transform = transform
        self.cls2idx = {c:i for i,c in enumerate(classes)}
        self.idx2cls = {i:c for c,i in self.cls2idx.items()}
        self.paths, self.labels = [], []
        for c in classes:
            cls_dir = os.path.join(DATA_DIR, c)
            for f in os.listdir(cls_dir):
                if f.lower().endswith(('.mp4','.avi')):
                    self.paths.append(os.path.join(cls_dir, f))
                    self.labels.append(self.cls2idx[c])
        self.num_classes = len(classes)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        cap = cv2.VideoCapture(self.paths[idx])
        frames = []
        while len(frames) < MAX_FRAMES and cap.isOpened():
            ret, frm = cap.read()
            if not ret: break
            frames.append(frm)
        cap.release()
        if not frames:
            clip = np.zeros((MAX_FRAMES,112,112,3), dtype=np.uint8)
        else:
            clip = frames if len(frames) >= MAX_FRAMES else frames + frames[:MAX_FRAMES-len(frames)]
            clip = np.stack(clip)
        return (self.transform(clip), self.labels[idx]) if self.transform else (clip, self.labels[idx])

# ----------------------
# 1) Build and split full dataset
total_classes = sorted([d for d in os.listdir(DATA_DIR)
                        if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f"Total classes: {len(total_classes)}.")
full_ds = GestureDataset(total_classes, transform=VideoTransform())
idxs = list(range(len(full_ds)))
tr_idx, va_idx = train_test_split(idxs, test_size=VAL_SIZE,
                                  stratify=full_ds.labels, random_state=42)
# sampler for imbalance
t_counts = np.bincount([full_ds.labels[i] for i in tr_idx])
wts = 1.0 / t_counts
train_sampler = WeightedRandomSampler([wts[full_ds.labels[i]] for i in tr_idx], len(tr_idx), replacement=True)
train_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(full_ds, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(va_idx), num_workers=4, pin_memory=True)

# ----------------------
# 2) Model, criterion, optimizer
device = DEVICE
model = torchvision.models.video.r3d_18(weights=torchvision.models.video.R3D_18_Weights.KINETICS400_V1)
model.fc = nn.Linear(model.fc.in_features, full_ds.num_classes)
model = nn.DataParallel(model).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(wts,dtype=torch.float).to(device))
optimizer = optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=LR_HEAD)
scaler = GradScaler()

# ----------------------
# 3) Quick head-only train to gauge F1
def train_loop(loader, ep):
    model.train(); total_loss=0
    for clips, labels in tqdm(loader, desc=f"Train {ep}"):
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast(device_type=DEVICE):
            loss = criterion(model(clips), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    print(f"Epoch {ep} Loss: {total_loss/len(loader):.4f}")
for e in range(EPOCH_HEAD1): train_loop(train_loader, e) 

# ----------------------
# 4) Initial val & compute F1
model.eval(); preds, trues = [], []
with torch.no_grad(), autocast(device_type=DEVICE):
    for clips, labels in val_loader:
        clips = clips.to(device)
        preds += model(clips).argmax(1).cpu().tolist()
        trues += labels.tolist()
_, _, f1_scores, _ = precision_recall_fscore_support(trues, preds, labels=list(range(full_ds.num_classes)), zero_division=0)
class_f1 = {full_ds.idx2cls[i]: f1_scores[i] for i in range(full_ds.num_classes)}
reliable = [w for w,v in class_f1.items() if v>0]
print(f"Keeping {len(reliable)} classes with F1>0")

# ----------------------
# 5) Reduced dataset
ds2 = GestureDataset(reliable, transform=VideoTransform())
idx2 = list(range(len(ds2)))
tr2, va2 = train_test_split(idx2, test_size=VAL_SIZE, stratify=ds2.labels, random_state=42)
c2 = np.bincount([ds2.labels[i] for i in tr2]); w2 = 1.0/c2
sampler2 = WeightedRandomSampler([w2[ds2.labels[i]] for i in tr2], len(tr2), replacement=True)
train_loader2 = DataLoader(ds2, batch_size=BATCH_SIZE, sampler=sampler2, num_workers=4, pin_memory=True)
val_loader2   = DataLoader(ds2, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(va2), num_workers=4, pin_memory=True)

# ----------------------
# 6) Final train
def reset_model(nc):
    m = torchvision.models.video.r3d_18(weights=torchvision.models.video.R3D_18_Weights.KINETICS400_V1)
    m.fc = nn.Linear(m.fc.in_features, nc)
    return nn.DataParallel(m).to(device)
model = reset_model(ds2.num_classes) 
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w2,dtype=torch.float).to(device))
optimizer = optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=LR_HEAD)
scaler = GradScaler()
for e in range(EPOCH_HEAD2): train_loop(train_loader2, e)
for p in model.parameters(): p.requires_grad=True
optimizer = optim.Adam(model.parameters(), lr=LR_FINE)
for e in range(EPOCH_HEAD2, EPOCH_HEAD2+EPOCH_FINE): train_loop(train_loader2, e)

# ----------------------
# 7) Final eval & report
model.eval(); preds2, trues2 = [], []
with torch.no_grad(), autocast(device_type=DEVICE):
    for clips, labels in val_loader2:
        clips = clips.to(device)
        preds2 += model(clips).argmax(1).cpu().tolist()
        trues2 += labels.tolist()
print("\nFinal Results on F1>0 Classes:")
print(classification_report(
    trues2,
    preds2,
    labels=list(range(ds2.num_classes)),
    target_names=[ds2.idx2cls[i] for i in range(ds2.num_classes)],
    digits=4,
    zero_division=0
))

# 8) Show per-word F1
_, _, f1_2, _ = precision_recall_fscore_support(trues2, preds2, labels=list(range(ds2.num_classes)), zero_division=0)
print("Trained words and F1:")
for i,w in ds2.idx2cls.items(): print(f"{w}: {f1_2[i]:.4f}")

# 9) Save weights & generate download link
model_path = '/kaggle/working/gesture_model_reliable.pth'
torch.save(model.state_dict(), model_path)
display(FileLink(model_path))


ModuleNotFoundError: No module named 'scipy'

In [ ]:
from IPython.display import FileLink
FileLink('gesture_model_reliable.pth')


In [14]:
model_path = '/kaggle/working/gesture_model.pth'
torch.save(model, model_path)
display(FileLink(model_path))

/kaggle/working/gesture_model.pth

In [15]:
!ls -lh /kaggle/working


total 383M
-rw-r--r-- 1 root root 128M Nov  5 04:23 gesture_model_full.pth
-rw-r--r-- 1 root root 128M Nov  5 04:26 gesture_model.pth
-rw-r--r-- 1 root root 128M Nov  5 04:21 gesture_model_reliable.pth
-rw-r--r-- 1 root root  310 Nov  5 01:24 state.db


In [16]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/gesture_model.pth'))


/kaggle/working/gesture_model.pth